# Practical 5 – Word Embedding and Word Prediction

**Aim:** Convert a sentence into word indices, create context-target training examples, and train a small neural network to predict a missing middle word.

Every important code line is explained using `# comments` directly inside the code. Run the cells from top to bottom.

## 1. Import libraries

In [ ]:
# NumPy handles the input and target arrays.
import numpy as np

# Pandas is imported as in the practical (not otherwise required in this code).
import pandas as pd

# TensorFlow provides tensor operations and the neural-network model.
import tensorflow as tf

# Import the sequential model container.
from tensorflow.keras.models import Sequential

# Embedding learns word vectors; Dense predicts a vocabulary word; Lambda averages context vectors.
from tensorflow.keras.layers import Embedding, Dense, Lambda

# Tokenizer converts words into integer indices.
from tensorflow.keras.preprocessing.text import Tokenizer

## 2. Tokenize the sentence

In [ ]:
# Example sentence used to create a small word-prediction dataset.
text = "I love deep learning and machine learning"

# Create a tokenizer that assigns an integer index to each unique word.
tokenizer = Tokenizer()

# Build the word-to-index dictionary from this sentence.
tokenizer.fit_on_texts([text])

# Convert the sentence into a list of integer word IDs.
sequence = tokenizer.texts_to_sequences([text])[0]

# Count unique words and reserve index 0 by adding 1.
vocab_size = len(tokenizer.word_index) + 1

# Display the word dictionary, integer sequence, and vocabulary size.
print("Word Index:", tokenizer.word_index)
print("Sequence:", sequence)
print("Vocabulary Size:", vocab_size)

## 3. Create context and target examples

In [ ]:
# X stores the four surrounding context words; y stores the missing middle word.
X = []
y = []

# Start at index 2 so two words exist before the target; stop before two words are missing after it.
for i in range(2, len(sequence) - 2):
    # Use two words before and two words after the current target as context.
    X.append([sequence[i-2], sequence[i-1], sequence[i+1], sequence[i+2]])

    # The current word is the target the model should learn to predict.
    y.append(sequence[i])

# Convert Python lists into NumPy arrays for model input.
X = np.array(X)
y = np.array(y)

# Display context IDs and their target word IDs.
print("Training context:", X)
print("Target Words:", y)

## 4. Build the word prediction model

In [ ]:
# Create a model whose layers run sequentially.
model = Sequential([
    Embedding(vocab_size, 10),  # Map each word ID to a learned vector of length 10.
    Lambda(lambda x: tf.reduce_mean(x, axis=1)),  # Average the four context-word vectors into one vector.
    Dense(vocab_size, activation='softmax')  # Produce a probability for each vocabulary word.
])

# Tell Keras that each training example contains 4 context word IDs.
# None means the number of examples in a batch can vary.
model.build(input_shape=(None, 4))

# Display the model layers and parameter counts.
model.summary()

## 5. Compile and train

In [ ]:
# Configure the optimizer, integer-label classification loss, and accuracy metric.
model.compile(
    optimizer='adam',                         # Adam updates the embedding and Dense weights.
    loss='sparse_categorical_crossentropy',   # Suitable because target words are integer IDs.
    metrics=['accuracy']                      # Track how often the predicted word is correct.
)

In [ ]:
# Train the model using the context IDs (X) and target word IDs (y).
model.fit(X, y, epochs=100, verbose=0)  # 100 passes; verbose=0 hides progress output.

# Confirm that training has finished.
print("Model training completed")

## 6. Predict and display words

In [ ]:
# Predict a probability distribution over vocabulary words for each context.
predictions = model.predict(X, verbose=0)

# Display the predicted and actual word for every training example.
for i in range(len(X)):
    predicted = np.argmax(predictions[i])  # Choose the vocabulary index with the highest probability.

    # Convert each context word ID back into its readable word.
    print("Context:", [tokenizer.index_word[w] for w in X[i]])

    # Convert the actual target ID back into its readable word.
    print("Actual word:", tokenizer.index_word[y[i]])

    # Convert the predicted ID into a word; use 'Unknown' if no word is mapped.
    print("Predicted word:", tokenizer.index_word.get(predicted, "Unknown"))

    # Print a blank line to separate examples.
    print()